In [4]:
from functools import partial
import numpy as np
import pandas as pd

from cwfc.data import load_cpu, build_targets, split_train_holdout
from cwfc.baselines import persistence_forecast, train_mean_forecast
from cwfc.models import damped_persistence_forecast, ridge_recursive_forecast, ses_forecast
from cwfc.validation import rolling_origin_forecasts
from cwfc.metrics import mae, rmse, relative_improvement, subset_metrics

cpu = load_cpu("../challenge_data/vm_cpu_readings.csv.gz")
targets = build_targets(cpu)
train, _ = split_train_holdout(targets)   # the holdout is deliberately discarded here
print(len(train))                          # 110

110


In [5]:
target_b = "aggregate_cpu_utilization"

candidates = {
    "last_value": persistence_forecast,
    "train_mean": train_mean_forecast,
    "damped_persistence": damped_persistence_forecast,
    "ridge_a1": partial(ridge_recursive_forecast, alpha=1.0),
    "ridge_a10": partial(ridge_recursive_forecast, alpha=10.0),
    "ridge_a100": partial(ridge_recursive_forecast, alpha=100.0),
    "ses": ses_forecast,
}

results_b = {name: rolling_origin_forecasts(train, target_b, fn)
             for name, fn in candidates.items()}

Ridge is kept only if its rolling-origin MAE is lower than that of damped_persistence and its peak MAE is not worse. Otherwise damped_persistence is the final model for aggregate_cpu_utilization. The ridge strength (alpha 1, 10, 100) is also chosen by rolling-origin MAE only.

In [6]:
peak_threshold = train[target_b].quantile(0.90)   # recomputed from train, never typed in
ref_last = results_b["last_value"]
ref_mean = results_b["train_mean"]
ref_damped = results_b["damped_persistence"]

rows = []
for name, r in results_b.items():
    peak = subset_metrics(r.actual, r.pred, r.actual >= peak_threshold)
    rows.append({
        "model": name,
        "MAE": mae(r.actual, r.pred),
        "RMSE": rmse(r.actual, r.pred),
        "vs_last_value": relative_improvement(r.actual, r.pred, ref_last.pred),
        "vs_train_mean": relative_improvement(r.actual, r.pred, ref_mean.pred),
        "vs_damped": relative_improvement(r.actual, r.pred, ref_damped.pred),
        "peak_MAE": peak["mae"],
    })
table_b = pd.DataFrame(rows).sort_values("MAE").reset_index(drop=True)
print(table_b.round(4))

                model     MAE    RMSE  vs_last_value  vs_train_mean  \
0  damped_persistence  0.2025  0.2419         0.2372         0.0310   
1          train_mean  0.2090  0.2468         0.2128         0.0000   
2          ridge_a100  0.2163  0.2566         0.1853        -0.0349   
3                 ses  0.2546  0.3051         0.0411        -0.2181   
4          last_value  0.2655  0.3174         0.0000        -0.2703   
5            ridge_a1  0.2745  0.3302        -0.0338        -0.3131   
6           ridge_a10  0.2870  0.3525        -0.0810        -0.3732   

   vs_damped  peak_MAE  
0     0.0000    0.3961  
1    -0.0320    0.4157  
2    -0.0680    0.3556  
3    -0.2571    0.3447  
4    -0.3109    0.2455  
5    -0.3552    0.3485  
6    -0.4171    0.3569  


**What this shows:** No candidate beats `damped_persistence` on rolling-origin error (MAE 0.2025, RMSE 0.2419). The best ridge (alpha 100) is 6.8% worse in MAE, and simple exponential smoothing (`ses`) is 25.7% worse (MAE 0.2546), only 4.1% better than `last_value` and about equal to `moving_avg_6`. SES gives a flat forecast and does not return to the mean, which hurts at longer steps. Ridge with alpha 1 and 10 is worse than `last_value`, a sign of overfitting on about 36 to 98 training rows with five overlapping features; stronger shrinkage helps only because it pulls the forecast toward the mean, which `train_mean` already provides. The pattern across all 11 candidates is that models which pull toward the long-run average do best, and extra complexity does not help. Ridge and SES have a lower peak MAE (0.345 to 0.357) than damped persistence (0.396), but they are not better overall, and the peak figures rest on only 11 distinct timestamps.

**Decision (rules fixed before running):** a candidate replaces `damped_persistence` only if its rolling-origin MAE is lower and its peak MAE is not worse. None does, so `damped_persistence` is the final model for `aggregate_cpu_utilization` and `last_value` is used for `active_vm_count`. No further models will be tested.